In [46]:
import pandas as pd

In [47]:

footfall_df = pd.read_csv('../datasets/raw/StationFootfall_2024_2025.csv')

footfall_df.rename(columns={'TravelDate' : 'date', 'DayOfWeek' : 'day_of_week', 'Station' : 'station','EntryTapCount' : 'entry_tap_count', 'ExitTapCount' : 'exit_tap_count'}, inplace=True)

footfall_df['date'] = pd.to_datetime(footfall_df['date'].astype(str), format="%Y%m%d")

footfall_df['baseline_entry'] = (
    footfall_df.groupby(['station', 'day_of_week'])['entry_tap_count']
      .transform('mean')
)

footfall_df['baseline_exit'] = (
    footfall_df.groupby(["station", 'day_of_week'])['exit_tap_count']
      .transform('mean')
)

footfall_df['overcrowding_index'] = (
    footfall_df['entry_tap_count'] / footfall_df['baseline_entry']
) * 100

print(footfall_df.head())

        date day_of_week          station  entry_tap_count  exit_tap_count  \
0 2024-01-01      Monday   Abbey Road DLR              395             375   
1 2024-01-01      Monday       Abbey Wood             5898            5963   
2 2024-01-01      Monday    Acton Central              609             474   
3 2024-01-01      Monday  Acton Main Line             1717            1710   
4 2024-01-01      Monday       Acton Town             2928            3334   

   baseline_entry  baseline_exit  overcrowding_index  
0      859.721154     812.134615           45.945130  
1    15872.230769   15026.711538           37.159238  
2     2093.817308    2051.269231           29.085632  
3     3833.223301    3772.378641           44.792590  
4     6588.625000    6825.192308           44.440228  


In [48]:
weather_df = pd.read_csv('../datasets/raw/open-meteo-51.49N0.49W24m.csv')
weather_df['date'] = pd.to_datetime(weather_df['time'], format="%Y-%m-%d")

In [49]:
df_combined = pd.merge(
    footfall_df,
    weather_df,
    on="date",
    how="left"
)

In [50]:
df_combined.to_csv('../datasets/raw/combined.csv', index=False)

In [51]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311447 entries, 0 to 311446
Data columns (total 26 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   date                             311447 non-null  datetime64[ns]
 1   day_of_week                      311447 non-null  object        
 2   station                          311447 non-null  object        
 3   entry_tap_count                  311447 non-null  int64         
 4   exit_tap_count                   311447 non-null  int64         
 5   baseline_entry                   311447 non-null  float64       
 6   baseline_exit                    311447 non-null  float64       
 7   overcrowding_index               311447 non-null  float64       
 8   time                             311447 non-null  object        
 9   temperature_2m_max (°C)          311447 non-null  float64       
 10  temperature_2m_min (°C)          311447 non-

In [52]:
df_combined['is_raining'] = (df_combined['rain_sum (mm)'] > 0.0).astype('int')

In [53]:
df_combined['is_raining']

0         1
1         1
2         1
3         1
4         1
         ..
311442    0
311443    0
311444    0
311445    0
311446    0
Name: is_raining, Length: 311447, dtype: int64

In [54]:
df_combined = df_combined.rename(columns={
    "date": "date",
    "day_of_week": "day_of_week",
    "station": "station",
    "entry_tap_count": "entries",
    "exit_tap_count": "exits",
    "baseline_entry": "baseline_entries",
    "baseline_exit": "baseline_exits",
    "overcrowding_index": "overcrowding",
    "time": "hour",
    "temperature_2m_max (°C)": "temp_max",
    "temperature_2m_min (°C)": "temp_min",
    "temperature_2m_mean (°C)": "temp_mean",
    "apparent_temperature_mean (°C)": "app_temp_mean",
    "apparent_temperature_max (°C)": "app_temp_max",
    "apparent_temperature_min (°C)": "app_temp_min",
    "wind_speed_10m_max (km/h)": "wind_max",
    "wind_gusts_10m_max (km/h)": "wind_gust_max",
    "wind_direction_10m_dominant (°)": "wind_dir",
    "rain_sum (mm)": "rain_mm",
    "sunshine_duration (s)": "sunshine_s",
    "daylight_duration (s)": "daylight_s",
    "sunrise (iso8601)": "sunrise",
    "sunset (iso8601)": "sunset",
    "precipitation_hours (h)": "precip_hours",
    "snowfall_sum (cm)": "snow_cm",
    "precipitation_sum (mm)": "precip_mm"
})

In [55]:
df_combined.to_csv('../datasets/raw/combined.csv', index=False)

In [56]:
df = pd.read_csv('../datasets/raw/combined.csv')

In [57]:
# convert date time
df['date'] = pd.to_datetime(df['date'])
df['hour'] = pd.to_datetime(df['hour'], format="%H:%M", errors='coerce').dt.hour

# useful time features
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# sunrise and sunset
df['sunrise_hour'] = pd.to_datetime(df['sunrise'], errors='coerce').dt.hour
df['sunset_hour'] = pd.to_datetime(df['sunset'], errors='coerce').dt.hour

# encode categories
df['station_code'] = df['station'].astype('category').cat.codes

# numerical day of week
if df['day_of_week'].dtype == object:
    df['day_of_week_code'] = df['day_of_week'].astype('category').cat.codes
else:
    df['day_of_week_code'] = df['day_of_week']


# missing vals
df_rf = df.fillna(0)
df = df_rf.copy()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# -----------------------------
# 2. Choose your target variable
# -----------------------------
# Predicting entries (you can switch to exit_tap_count or overcrowding_index)
target = "entries"

# -----------------------------
# 3. Select features
# -----------------------------
features = df.drop(columns=[
    "date",
    "entries",
    "exits",
    "sunrise",
    "sunset",
    "hour"        # optional: remove if it's a datetime string
], errors="ignore")


X = features
y = df[target]

# -----------------------------
# 4. Identify categorical columns
# -----------------------------
categorical_cols = ["day_of_week", "station"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# -----------------------------
# 5. Preprocessing
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# -----------------------------
# 6. Build the Random Forest pipeline
# -----------------------------
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

# -----------------------------
# 7. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# 8. Fit the model
# -----------------------------
model.fit(X_train, y_train)

# -----------------------------
# 9. Evaluate
# -----------------------------
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("MAE:", mae)
print("R²:", r2)

# -----------------------------
# 10. Feature importances
# -----------------------------
rf = model.named_steps["rf"]
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
all_feature_names = list(cat_feature_names) + numeric_cols

importances = pd.Series(rf.feature_importances_, index=all_feature_names)
print(importances.sort_values(ascending=False).head(20))